In [1]:
import pandas as pd

In [3]:
df_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')

In [4]:
df_identity = pd.read_csv('ieee-fraud-detection/train_identity.csv')

In [5]:
df = df_transaction.merge(df_identity, on="TransactionID", how="left")

In [6]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [7]:
print(df_transaction.shape)
print(df_identity.shape)
print(df.shape)
print(df["isFraud"].mean())

(590540, 394)
(144233, 41)
(590540, 434)
0.03499000914417313


In [8]:
audit = pd.DataFrame({
    "column": df.columns,
    "missing_pct": df.isna().mean().mul(100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

In [9]:
audit.to_csv("ieee_cis_merged_column_audit.csv", index=False)

In [10]:
audit.head()

,column,missing_pct,n_unique
id_24,id_24,99.20,12
id_25,id_25,99.13,341
id_26,id_26,99.13,95
id_21,id_21,99.13,490
id_08,id_08,99.13,94


In [11]:
pilot_pool = df[
    df["DeviceType"].notna() |
    df["DeviceInfo"].notna() |
    df["id_31"].notna() |
    df["id_30"].notna()
]

In [12]:
pilot_pool.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
4,2987004,0,86506,50.000,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
8,2987008,0,86535,15.000,H,2803,100.0,150.0,visa,226.0,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
10,2987010,0,86549,75.887,C,16496,352.0,117.0,mastercard,134.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
11,2987011,0,86555,16.495,C,4461,375.0,185.0,mastercard,224.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
16,2987016,0,86620,30.000,H,1790,555.0,150.0,visa,226.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS


In [13]:
fraud_cases = pilot_pool[pilot_pool["isFraud"] == 1].sample(25, random_state=42)
nonfraud_cases = pilot_pool[pilot_pool["isFraud"] == 0].sample(25, random_state=42)

In [14]:
pilot_cases = pd.concat([fraud_cases, nonfraud_cases]).sample(frac=1, random_state=42)

In [15]:
identity_coverage = df[["DeviceType", "DeviceInfo", "id_30", "id_31", "id_33", "id_34", "id_35", "id_36", "id_37", "id_38"]].notna().mean().mul(100).round(2)

In [16]:
print(identity_coverage)

DeviceType    23.84
DeviceInfo    20.09
id_30         13.13
id_31         23.75
id_33         12.41
id_34         13.18
id_35         23.87
id_36         23.87
id_37         23.87
id_38         23.87
dtype: float64


In [17]:
pilot_pool = df[df["DeviceType"].notna() | df["DeviceInfo"].notna()]

In [18]:
print(pilot_pool.shape)
print(pilot_pool["isFraud"].mean())
print(pilot_pool["isFraud"].value_counts())

(140855, 434)
0.07959958822902985
isFraud
0    129643
1     11212
Name: count, dtype: int64


In [19]:
pilot_pool = df[
    (df["DeviceType"].notna() | df["DeviceInfo"].notna()) &
    df["card1"].notna() &
    df["TransactionAmt"].notna()
]

In [20]:
print(pilot_pool.shape)
print(pilot_pool["isFraud"].mean())
print(pilot_pool["isFraud"].value_counts())

(140855, 434)
0.07959958822902985
isFraud
0    129643
1     11212
Name: count, dtype: int64


In [21]:
fraud_cases = pilot_pool[pilot_pool["isFraud"] == 1].sample(25, random_state=42)
nonfraud_cases = pilot_pool[pilot_pool["isFraud"] == 0].sample(25, random_state=42)

pilot_cases = (
    pd.concat([fraud_cases, nonfraud_cases])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

In [22]:
print(pilot_cases.shape)
print(pilot_cases["isFraud"].value_counts())

(50, 434)
isFraud
1    25
0    25
Name: count, dtype: int64


In [23]:
pilot_cases.to_csv("fraudops_pilot_50_raw.csv", index=False)

In [24]:
import json
import pandas as pd

def safe_value(x):
    if pd.isna(x):
        return None
    if hasattr(x, "item"):
        return x.item()
    return x

cases = []

for i, row in pilot_cases.iterrows():
    case = {
        "case_id": f"CASE_{i+1:04d}",
        "transaction_id": int(row["TransactionID"]),
        "alert_type": "Card-not-present transaction risk review",

        # Keep this only for evaluation. Do not show to agent.
        "ground_truth_is_fraud": int(row["isFraud"]),

        "visible_case_summary": {
            "transaction_amount": safe_value(row["TransactionAmt"]),
            "product_code": safe_value(row["ProductCD"]),
            "transaction_time_delta": safe_value(row["TransactionDT"]),
            "card_brand": safe_value(row["card4"]),
            "card_type": safe_value(row["card6"]),
            "purchaser_email_domain": safe_value(row["P_emaildomain"]),
            "recipient_email_domain": safe_value(row["R_emaildomain"]),
            "billing_region_proxy": safe_value(row["addr1"]),
            "country_proxy": safe_value(row["addr2"]),
            "device_type": safe_value(row["DeviceType"]),
            "device_info": safe_value(row["DeviceInfo"]),
            "os_info": safe_value(row["id_30"]),
            "browser_info": safe_value(row["id_31"]),
            "screen_resolution": safe_value(row["id_33"])
        },

        "available_tools": [
            "get_transaction_details",
            "get_card_history",
            "get_email_domain_profile",
            "get_device_history",
            "get_velocity_summary",
            "get_identity_match_summary"
        ],

        "required_checks": [
            "transaction_amount_check",
            "card_history_check",
            "email_domain_check",
            "device_history_check",
            "velocity_check",
            "identity_consistency_check"
        ],

        "expected_output_fields": [
            "disposition",
            "risk_indicators",
            "evidence_used",
            "missing_evidence",
            "final_case_note"
        ]
    }

    cases.append(case)

with open("fraudops_bench_v0_cases.jsonl", "w") as f:
    for case in cases:
        f.write(json.dumps(case) + "\n")

In [31]:
import pandas as pd
import numpy as np

# df should be your merged IEEE-CIS dataframe
# Make sure df is sorted by TransactionDT
df = df.sort_values("TransactionDT").reset_index(drop=True)


def safe_json_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    return x


def get_transaction_details(transaction_id):
    row = df[df["TransactionID"] == transaction_id]

    if row.empty:
        return {"error": "TransactionID not found"}

    row = row.iloc[0]

    fields = [
        "TransactionID", "TransactionDT", "TransactionAmt", "ProductCD",
        "card1", "card2", "card3", "card4", "card5", "card6",
        "addr1", "addr2", "dist1", "dist2",
        "P_emaildomain", "R_emaildomain",
        "DeviceType", "DeviceInfo", "id_30", "id_31", "id_33",
        "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"
    ]

    return {col: safe_json_value(row[col]) for col in fields if col in df.columns}


def get_card_history(transaction_id, max_history=20):
    row = df[df["TransactionID"] == transaction_id]

    if row.empty:
        return {"error": "TransactionID not found"}

    row = row.iloc[0]
    card1 = row["card1"]
    current_time = row["TransactionDT"]

    hist = df[
        (df["card1"] == card1) &
        (df["TransactionDT"] < current_time)
    ].sort_values("TransactionDT").tail(max_history)

    recent_cols = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "ProductCD",
        "card4",
        "card6",
        "P_emaildomain",
        "R_emaildomain",
        "DeviceType",
        "DeviceInfo"
    ]

    recent_cols = [col for col in recent_cols if col in hist.columns]

    return {
        "card_proxy": safe_json_value(card1),
        "prior_transaction_count": int(len(hist)),

        # Analyst-facing aggregate history
        "confirmed_prior_fraud_count": int(hist["isFraud"].sum()) if len(hist) else 0,
        "confirmed_prior_fraud_rate": float(hist["isFraud"].mean()) if len(hist) else None,

        "avg_prior_amount": float(hist["TransactionAmt"].mean()) if len(hist) else None,
        "max_prior_amount": float(hist["TransactionAmt"].max()) if len(hist) else None,
        "current_amount_vs_avg_prior_amount": (
            float(row["TransactionAmt"] / hist["TransactionAmt"].mean())
            if len(hist) and hist["TransactionAmt"].mean() not in [0, None]
            else None
        ),

        # No raw isFraud exposed here
        "recent_transactions": hist[recent_cols].map(safe_json_value).to_dict("records")
    }


def get_email_domain_profile(transaction_id):
    row = df[df["TransactionID"] == transaction_id]

    if row.empty:
        return {"error": "TransactionID not found"}

    row = row.iloc[0]
    p_domain = row["P_emaildomain"]
    r_domain = row["R_emaildomain"]

    p_hist = df[df["P_emaildomain"] == p_domain] if pd.notna(p_domain) else pd.DataFrame()
    r_hist = df[df["R_emaildomain"] == r_domain] if pd.notna(r_domain) else pd.DataFrame()

    return {
        "purchaser_email_domain": safe_json_value(p_domain),
        "recipient_email_domain": safe_json_value(r_domain),
        "domains_match": bool(p_domain == r_domain) if pd.notna(p_domain) and pd.notna(r_domain) else None,
        "p_domain_transaction_count": int(len(p_hist)),
        "p_domain_fraud_rate": float(p_hist["isFraud"].mean()) if len(p_hist) else None,
        "r_domain_transaction_count": int(len(r_hist)),
        "r_domain_fraud_rate": float(r_hist["isFraud"].mean()) if len(r_hist) else None
    }


def get_device_history(transaction_id, max_history=20):
    row = df[df["TransactionID"] == transaction_id]

    if row.empty:
        return {"error": "TransactionID not found"}

    row = row.iloc[0]
    device_info = row["DeviceInfo"]
    device_type = row["DeviceType"]
    current_time = row["TransactionDT"]

    if pd.isna(device_info):
        return {
            "device_info": None,
            "device_type": safe_json_value(device_type),
            "message": "No DeviceInfo available for this transaction"
        }

    hist = df[
        (df["DeviceInfo"] == device_info) &
        (df["TransactionDT"] < current_time)
    ].sort_values("TransactionDT").tail(max_history)

    recent_cols = [
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "ProductCD",
        "card4",
        "card6",
        "P_emaildomain",
        "R_emaildomain",
        "DeviceType",
        "DeviceInfo"
    ]

    recent_cols = [col for col in recent_cols if col in hist.columns]

    return {
        "device_info": safe_json_value(device_info),
        "device_type": safe_json_value(device_type),
        "prior_device_transaction_count": int(len(hist)),

        # Analyst-facing aggregate history
        "confirmed_prior_device_fraud_count": int(hist["isFraud"].sum()) if len(hist) else 0,
        "confirmed_prior_device_fraud_rate": float(hist["isFraud"].mean()) if len(hist) else None,

        "avg_prior_device_amount": float(hist["TransactionAmt"].mean()) if len(hist) else None,
        "max_prior_device_amount": float(hist["TransactionAmt"].max()) if len(hist) else None,
        "current_amount_vs_avg_prior_device_amount": (
            float(row["TransactionAmt"] / hist["TransactionAmt"].mean())
            if len(hist) and hist["TransactionAmt"].mean() not in [0, None]
            else None
        ),

        # No raw isFraud exposed here
        "recent_device_transactions": hist[recent_cols].map(safe_json_value).to_dict("records")
    }


def get_velocity_summary(transaction_id):
    row = df[df["TransactionID"] == transaction_id]

    if row.empty:
        return {"error": "TransactionID not found"}

    row = row.iloc[0]

    c_cols = [f"C{i}" for i in range(1, 15) if f"C{i}" in df.columns]
    d_cols = [f"D{i}" for i in range(1, 16) if f"D{i}" in df.columns]

    c_percentiles = {}
    d_missing_count = int(row[d_cols].isna().sum())

    for col in c_cols:
        val = row[col]
        if pd.isna(val):
            c_percentiles[col] = None
        else:
            c_percentiles[col] = float((df[col] <= val).mean())

    high_c_count = sum(
        1 for v in c_percentiles.values()
        if v is not None and v >= 0.95
    )

    return {
        "count_feature_summary": {
            "high_count_feature_count_95th_pct": high_c_count,
            "max_C_value": safe_json_value(row[c_cols].max()),
            "mean_C_value": safe_json_value(row[c_cols].mean())
        },
        "time_delta_feature_summary": {
            "max_D_value": safe_json_value(row[d_cols].max()),
            "mean_D_value": safe_json_value(row[d_cols].mean()),
            "missing_D_count": d_missing_count
        }
    }

def get_identity_match_summary(transaction_id):
    row = df[df["TransactionID"] == transaction_id]

    if row.empty:
        return {"error": "TransactionID not found"}

    row = row.iloc[0]

    m_cols = [f"M{i}" for i in range(1, 10) if f"M{i}" in df.columns]
    id_flag_cols = ["id_12", "id_15", "id_16", "id_28", "id_29", "id_35", "id_36", "id_37", "id_38"]
    id_flag_cols = [col for col in id_flag_cols if col in df.columns]

    return {
        "match_features": {col: safe_json_value(row[col]) for col in m_cols},
        "identity_flags": {col: safe_json_value(row[col]) for col in id_flag_cols},
        "device_type": safe_json_value(row.get("DeviceType")),
        "device_info": safe_json_value(row.get("DeviceInfo")),
        "os_info": safe_json_value(row.get("id_30")),
        "browser_info": safe_json_value(row.get("id_31")),
        "screen_resolution": safe_json_value(row.get("id_33"))
    }

In [32]:
case_txn = int(pilot_cases.iloc[0]["TransactionID"])

In [33]:
print(get_transaction_details(case_txn))
print(get_card_history(case_txn))
print(get_email_domain_profile(case_txn))
print(get_device_history(case_txn))
print(get_velocity_summary(case_txn))
print(get_identity_match_summary(case_txn))

{'TransactionID': 3235903, 'TransactionDT': 5936800, 'TransactionAmt': 150.0, 'ProductCD': 'H', 'card1': 7919, 'card2': 194.0, 'card3': 150.0, 'card4': 'mastercard', 'card5': 166.0, 'card6': 'debit', 'addr1': 225.0, 'addr2': 87.0, 'dist1': None, 'dist2': None, 'P_emaildomain': 'gmail.com', 'R_emaildomain': 'gmail.com', 'DeviceType': 'desktop', 'DeviceInfo': 'Windows', 'id_30': 'Windows 7', 'id_31': 'chrome 63.0', 'id_33': '1600x900', 'M1': None, 'M2': None, 'M3': None, 'M4': None, 'M5': None, 'M6': None, 'M7': None, 'M8': None, 'M9': None}
{'card_proxy': 7919, 'prior_transaction_count': 20, 'confirmed_prior_fraud_count': 0, 'confirmed_prior_fraud_rate': 0.0, 'avg_prior_amount': 97.70750000000001, 'max_prior_amount': 554.0, 'current_amount_vs_avg_prior_amount': 1.5351943300156077, 'recent_transactions': [{'TransactionID': 3235524, 'TransactionDT': 5928923, 'TransactionAmt': 209.95, 'ProductCD': 'W', 'card4': 'mastercard', 'card6': 'debit', 'P_emaildomain': 'gmail.com', 'R_emaildomain': 

In [40]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

True

In [41]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [42]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Say hello in JSON only."
)

In [43]:
print(response.text)

{
  "message": "hello"
}
